# Task 1: Data Preprocessing

This notebook implements a **Named Entity Recognition (NER)** pipeline using a **BiLSTM + CRF** model built with MindSpore. The dataset used is the CoNLL-2003 NER benchmark.

The four main tasks are:
1. **Data Preprocessing** — load raw text, build vocabulary, encode labels, and prepare tensors
2. **Model Construction** — implement the CRF layer (score, normalizer, Viterbi decoder) and the BiLSTM+CRF model
3. **Model Training & Saving** — train with mini-batches, save checkpoint, export to AIR format
4. **Model Prediction** — load the saved model and run inference on test data

## Section 1.1 — Define the Data Loading Function

The CoNLL-2003 file format stores one token per line with four space-separated columns:
`WORD  POS  CHUNK  NER-LABEL`. Sentences are separated by blank lines.

The `load_file` function reads this format and returns two parallel lists:
- `train_sentences` — a list of token lists (one list per sentence)
- `train_labels`   — a list of NER label lists (aligned with sentences)

In [1]:
import os

dataroot_path = "./conll2003"

def load_file(path):
    """
    Load a CoNLL-2003 formatted file and return parallel lists of
    sentences (token lists) and their corresponding NER label lists.

    Each non-blank line has the format: WORD POS CHUNK LABEL
    Blank lines act as sentence boundaries.

    Args:
        path (str): Relative path to the data file (appended to dataroot_path).

    Returns:
        train_sentences (list[list[str]]): Tokenised sentences.
        train_labels    (list[list[str]]): Corresponding NER tag sequences.
    """
    train_sentences = []
    train_labels = []

    with open(dataroot_path + path) as f:
        sentence = []
        labels = []
        for line in f:
            line = line.strip()
            if line:
                # Unpack all four CoNLL columns; we only need WORD and LABEL
                word, pos, chunk, label = line.split()
                sentence.append(word)   # collect the token
                labels.append(label)    # collect its NER tag
            else:
                # Blank line signals the end of a sentence
                if sentence:            # avoid appending empty sentences
                    train_sentences.append(sentence)
                    train_labels.append(labels)
                sentence = []
                labels = []

        # Handle files that do not end with a trailing blank line
        if sentence:
            train_sentences.append(sentence)
            train_labels.append(labels)

    return train_sentences, train_labels

In [2]:
pip install mindspore==2.0.0

Looking in indexes: http://repo.myhuaweicloud.com/repository/pypi/simple/

[notice] A new release of pip available: 22.2.2 -> 24.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Load the training split from the CoNLL-2003 dataset
train_sentences, train_labels = load_file("/train.txt")

In [4]:
# Sanity-check: print the first 4 sentences and their NER labels
train_sentences[:4], train_labels[:4]

([['-DOCSTART-'],
  ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'],
  ['Peter', 'Blackburn'],
  ['BRUSSELS', '1996-08-22']],
 [['O'],
  ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O'],
  ['B-PER', 'I-PER'],
  ['B-LOC', 'O']])

## Section 1.2 — Vocabulary Construction and Label Encoding

To feed text into a neural network we need two mappings:

| Mapping | Purpose |
|---|---|
| `word_to_idx` | Maps every unique token to an integer index. Index 0 is reserved for `<pad>`. |
| `tag_to_idx`  | Maps each NER tag to a fixed integer (9 tags for CoNLL-2003 IOB2 scheme). |

We build `word_to_idx` from the training data so it covers all tokens the model will be trained on.

In [5]:
def get_dataset(sentences, labels):
    """
    Zip sentences and labels into a list of (sentence, label) tuples,
    which is the standard format expected by the rest of the pipeline.
    """
    dataset = []
    for i in range(len(sentences)):
        data_sample = tuple((sentences[i], labels[i]))
        dataset.append(data_sample)
    return dataset

# Build the full training dataset (all 14 987 sentences)
training_data = get_dataset(train_sentences, train_labels)

# Uncomment the line below to use only the first 1 000 sentences for a quick run
# training_data = get_dataset(train_sentences[:1000], train_labels)

len(training_data)

14987

In [6]:
# ── Vocabulary construction ──────────────────────────────────────────────────
# Index 0 is always reserved for the padding token <pad>.
# Every other token is assigned the next available integer in encounter order.
word_to_idx = {}
word_to_idx['<pad>'] = 0

for sentence, tags in training_data:
    for word in sentence:
        if word not in word_to_idx:
            word_to_idx[word] = len(word_to_idx)   # assign next available index

# ── Label encoding ───────────────────────────────────────────────────────────
# CoNLL-2003 uses the IOB2 tagging scheme with 4 entity types:
#   PER (person), ORG (organisation), LOC (location), MISC (miscellaneous)
# B- = beginning of an entity span, I- = inside a span, O = outside any entity
tag_to_idx = {
    "B-PER": 0, "I-PER": 1,
    "B-ORG": 2, "I-ORG": 3,
    "B-LOC": 4, "I-LOC": 5,
    "B-MISC": 6, "I-MISC": 7,
    "O": 8
}

In [7]:
# The vocabulary should contain 23 625 unique tokens (including <pad>)
len(word_to_idx)

23625

In [8]:
# Inspect the first 5 vocabulary entries — Method 1: via list(dict.items())
items = list(word_to_idx.items())
items[:5]

[('<pad>', 0), ('-DOCSTART-', 1), ('EU', 2), ('rejects', 3), ('German', 4)]

In [9]:
# Inspect the first 5 vocabulary entries — Method 2: via OrderedDict
# (Both methods produce identical output; shown here for completeness)
from collections import OrderedDict
ordered_dict_items = OrderedDict(word_to_idx)
list(ordered_dict_items.items())[:5]

[('<pad>', 0), ('-DOCSTART-', 1), ('EU', 2), ('rejects', 3), ('German', 4)]

## Section 1.3 — Data Format Conversion (Padding & Truncation)

Neural networks require fixed-length inputs within a batch. We use a maximum sequence length of **10 tokens**:

- **Sequences shorter than 10**: pad with `<pad>` (index 0) and the `O` label.
- **Sequences longer than 10**: truncate to the first 10 tokens.

`prepare_sequence` returns three MindSpore `int64` Tensors:

| Tensor | Shape | Description |
|---|---|---|
| `seq_outputs` | `(N, max_len)` | Token index sequences |
| `label_outputs` | `(N, max_len)` | Label index sequences |
| `seq_length` | `(N,)` | True (pre-padding) length of each sequence |

In [10]:
import mindspore

def prepare_sequence(seqs, word_to_idx, tag_to_idx, max_len=10):
    """
    Convert a list of (sentence, label) tuples into padded/truncated
    MindSpore Tensors ready for model input.

    Args:
        seqs        : list of (sentence: list[str], tags: list[str]) tuples
        word_to_idx : dict mapping tokens  -> integer indices
        tag_to_idx  : dict mapping NER tags -> integer indices
        max_len     : fixed sequence length (default 10)

    Returns:
        Tensor (int64) of shape (N, max_len)  -- token index sequences
        Tensor (int64) of shape (N, max_len)  -- label index sequences
        Tensor (int64) of shape (N,)          -- true sequence lengths
    """
    seq_outputs, label_outputs, seq_length = [], [], []

    for seq, tag in seqs:
        if max_len > len(seq):
            # ── Short sequence: record true length, then pad to max_len ──────
            seq_length.append(len(seq))
            idxs   = [word_to_idx[w] for w in seq]
            labels = [tag_to_idx[t]  for t in tag]
            # Pad token indices with 0 (<pad>) and labels with O (index 8)
            idxs.extend(  [word_to_idx['<pad>'] for _ in range(max_len - len(seq))])
            labels.extend([tag_to_idx['O']      for _ in range(max_len - len(seq))])
        else:
            # ── Long sequence: truncate to max_len ───────────────────────────
            seq_length.append(max_len)
            idxs   = [word_to_idx[w] for w in seq[:max_len]]
            labels = [tag_to_idx[t]  for t in tag[:max_len]]

        seq_outputs.append(idxs)
        label_outputs.append(labels)

    return (
        mindspore.Tensor(seq_outputs,   mindspore.int64),
        mindspore.Tensor(label_outputs, mindspore.int64),
        mindspore.Tensor(seq_length,    mindspore.int64),
    )

In [11]:
# Verify shapes on the first 4 training samples.
# Expected: data=(4,10), label=(4,10), seq_length=(4,)
data, label, seq_length = prepare_sequence(training_data[:4], word_to_idx, tag_to_idx)
data.shape, label.shape, seq_length.shape

((4, 10), (4, 10), (4,))

---
# Task 2: Model Construction

We build a **BiLSTM + CRF** model for sequence labelling. The architecture is:

```
Input tokens
    └─► nn.Embedding          (vocab_size → embedding_dim)
            └─► nn.LSTM        (bidirectional, embedding_dim → hidden_dim)
                    └─► nn.Dense  (hidden_dim → num_tags)   ← emission scores
                            └─► CRF layer                   ← NLL loss / Viterbi decode
```

The CRF layer models dependencies between adjacent output labels, which is crucial for NER because entity tags are not independent (e.g. `I-PER` should only follow `B-PER` or `I-PER`).

## Section 2.1 — CRF Score: Computing the Score for the Ground-Truth Label Sequence

### Mathematical Background

For input sequence $x$ and label sequence $y$, the CRF score is defined as:

$$\text{Score}(x,y) = \sum_i \; h_i[y_i] \;+\; \mathbf{P}_{y_{i-1},\, y_i} \qquad (3)$$

where:
- $h_i[y_i]$ is the **emission score** — the value in the BiLSTM output at position $i$ for tag $y_i$
- $\mathbf{P}_{y_{i-1}, y_i}$ is the **transition score** from tag $y_{i-1}$ to tag $y_i$
- We also add a **start transition** $\mathbf{s}[y_0]$ and an **end transition** $\mathbf{e}[y_{\text{last}}]$

The `mask` tensor ensures that padding positions do not contribute to the score.

In [12]:
def compute_score(emissions, tags, seq_ends, mask, trans, start_trans, end_trans):
    """
    Compute the CRF score for the ground-truth tag sequences (the numerator
    of the log-likelihood).

    Args:
        emissions   : Tensor (seq_length, batch_size, num_tags)  — BiLSTM outputs
        tags        : Tensor (seq_length, batch_size)             — ground-truth labels
        seq_ends    : Tensor (batch_size,)                        — index of last real token
        mask        : Tensor (seq_length, batch_size)             — 1 for real tokens, 0 for pad
        trans       : Tensor (num_tags, num_tags)                 — transition matrix P
        start_trans : Tensor (num_tags,)                          — start transition vector
        end_trans   : Tensor (num_tags,)                          — end   transition vector

    Returns:
        score : Tensor (batch_size,) — the CRF score for each sample
    """
    seq_length, batch_size = tags.shape
    mask = mask.astype(emissions.dtype)

    # Initialise score with the start transition into the first tag
    # shape: (batch_size,)
    score = start_trans[tags[0]]

    # Add the emission score for the first token
    # For each sample b, pick the emission at position 0 for the correct tag
    # shape: (batch_size,)
    score += emissions[0, mnp.arange(batch_size), tags[0]]

    for i in range(1, seq_length):
        # Transition score from tag at (i-1) to tag at i
        # Multiply by mask[i] so padding positions contribute 0
        # shape: (batch_size,)
        score += trans[tags[i - 1], tags[i]] * mask[i]

        # Emission score for position i at the correct tag
        # Again gated by mask[i]
        # shape: (batch_size,)
        score += emissions[i, mnp.arange(batch_size), tags[i]] * mask[i]

    # Add the end transition from the last *real* tag of each sequence
    # seq_ends holds the index of the last non-pad position per sample
    # shape: (batch_size,)
    last_tags = tags[seq_ends, mnp.arange(batch_size)]
    score += end_trans[last_tags]

    return score

## Section 2.2 — CRF Normalizer: Log-Sum-Exp Over All Possible Label Sequences

### Mathematical Background

The CRF loss is:

$$\text{Loss} = \log \sum_{y' \in Y} \exp(\text{Score}(x, y')) - \text{Score}(x, y) \qquad (5)$$

The first term — the **Normalizer** — is the log-sum-exp over *all* possible tag sequences. Brute-force enumeration is $|T|^n$, which is intractable. Instead we use **dynamic programming**:

$$\log \sum_{y'_{0,i}} \exp(\text{Score}_i) = \log \sum_{y'_{0,i-1}} \exp(\text{Score}_{i-1}) + h_i + \mathbf{P} \qquad (7)$$

At each step we maintain a score vector of shape `(batch_size, num_tags)` where entry $[b, t]$ holds the log-sum-exp of all paths ending in tag $t$ at the current position for sample $b$.

In [13]:
def compute_normalizer(emissions, mask, trans, start_trans, end_trans):
    """
    Compute the CRF normalizer Z(x) = log Σ_{y'} exp(Score(x, y'))
    using dynamic programming (forward algorithm).

    Args:
        emissions   : Tensor (seq_length, batch_size, num_tags)
        mask        : Tensor (seq_length, batch_size)  — 1=real token, 0=pad
        trans       : Tensor (num_tags, num_tags)       — transition matrix P
        start_trans : Tensor (num_tags,)                — start transitions
        end_trans   : Tensor (num_tags,)                — end   transitions

    """
    seq_length = emissions.shape[0]

    # Initialise DP table: start transition + first emission
    # shape: (batch_size, num_tags)
    score = start_trans + emissions[0]

    for i in range(1, seq_length):
        # Expand score for broadcasting over next-tag dimension
        # shape: (batch_size, num_tags, 1)
        broadcast_score = score.expand_dims(2)

        # Expand emission for broadcasting over current-tag dimension
        # shape: (batch_size, 1, num_tags)
        broadcast_emissions = emissions[i].expand_dims(1)

        # Combine: for every (prev_tag, next_tag) pair compute the partial score
        # next_score[b, j, k] = score[b, j] + trans[j, k] + emission[b, k]
        # shape: (batch_size, num_tags, num_tags)
        next_score = broadcast_score + trans + broadcast_emissions

        # Collapse over the previous-tag dimension using numerically stable
        # log-sum-exp (subtract max before exp, add back after log)
        # shape: (batch_size, num_tags)
        next_score_max = next_score.max()
        next_score = mnp.log(mnp.sum(mnp.exp(next_score - next_score_max), axis=1)) + next_score_max

        # Only update score where the token is real (mask == 1)
        # shape: (batch_size, num_tags)
        score = mnp.where(mask[i].expand_dims(1), next_score, score)

    # Add end transition probabilities
    # shape: (batch_size, num_tags)
    score += end_trans

    # Final log-sum-exp over all tags to get the partition function
    # shape: (batch_size,)
    return mnp.log(mnp.sum(mnp.exp(score), axis=1))

## Section 2.3 — Viterbi Decoding

During **inference** we want the single highest-scoring tag sequence rather than the partition function. We use the **Viterbi algorithm**, which is a DP variant of the forward algorithm that tracks, at each position, which previous tag produced the best score.

The recurrence is:
$$P_{0,i} = \max(P_{0,\,i-1}) + P_{i-1,\,i}$$

After the forward pass we back-track through the saved `history` indices to recover the best path.

> **Note**: The back-tracking step (`post_decode`) is intentionally kept outside the `nn.Cell` because MindSpore's static graph mode does not support the dynamic Python constructs needed there.

In [14]:
def viterbi_decode(emissions, mask, trans, start_trans, end_trans):
    """
    Forward pass of the Viterbi algorithm.
    Returns the best-path scores and the per-step back-pointer history.

    Returns:
        score   : Tensor (batch_size, num_tags) — best scores at the final step
        history : tuple of Tensors              — argmax back-pointers for each step
    """
    seq_length = mask.shape[0]

    # Initialise with start transitions + first emission
    score = start_trans + emissions[0]
    history = ()

    for i in range(1, seq_length):
        broadcast_score    = score.expand_dims(2)        # (batch, num_tags, 1)
        broadcast_emission = emissions[i].expand_dims(1) # (batch, 1, num_tags)
        next_score = broadcast_score + trans + broadcast_emission

        # Save the best previous tag index for back-tracking
        indices = next_score.argmax(axis=1)  # (batch_size, num_tags)
        history += (indices,)

        # Keep only the max score for the next iteration
        next_score = next_score.max(axis=1)  # (batch_size, num_tags)
        score = mnp.where(mask[i].expand_dims(1), next_score, score)

    score += end_trans  # add end transition
    return score, history


def post_decode(score, history, seq_length):
    """
    Back-track through the Viterbi history to recover the best tag sequence
    for every sample in the batch.

    Returns:
        best_tags_list : list[list[int]] — predicted tag index sequences
    """
    batch_size = seq_length.shape[0]
    seq_ends   = seq_length - 1
    best_tags_list = []

    for idx in range(batch_size):
        # Find the best tag at the last real position
        best_last_tag = score[idx].argmax(axis=0)
        best_tags = [int(best_last_tag.asnumpy())]

        # Walk backwards through the history back-pointers
        for hist in reversed(history[:seq_ends[idx]]):
            best_last_tag = hist[idx][best_tags[-1]]
            best_tags.append(int(best_last_tag.asnumpy()))

        # Reverse to get the sequence in correct (left-to-right) order
        best_tags.reverse()
        best_tags_list.append(best_tags)

    return best_tags_list

## Section 2.4 — Assembling the Full CRF Layer

We now wrap the score and normalizer functions into a MindSpore `nn.Cell`.

**Learnable parameters:**
| Parameter | Shape | Description |
|---|---|---|
| `start_transitions` | `(num_tags,)` | Log-probability of a sequence starting with tag $t$ |
| `end_transitions`   | `(num_tags,)` | Log-probability of a sequence ending   with tag $t$ |
| `transitions`       | `(num_tags, num_tags)` | Log-probability of tag $i$ followed by tag $j$ |

The `construct` method routes to either the training forward pass (`_forward`) or the Viterbi decoder (`_decode`) depending on whether `tags` is provided.

In [15]:
import mindspore
import mindspore.nn as nn
import mindspore.numpy as mnp
from mindspore import Parameter
from mindspore.common.initializer import initializer, Uniform


def sequence_mask(seq_length, max_length, batch_first=False):
    """
    Generate a boolean mask tensor from sequence lengths.

    For each sample b and position i:
        mask[b, i] = 1  if i < seq_length[b]  (real token)
        mask[b, i] = 0  otherwise              (padding)

    Args:
        seq_length  : Tensor (batch_size,) — true lengths
        max_length  : int — padded sequence length
        batch_first : bool — if True return (batch, seq); else (seq, batch)

    Returns:
        int64 Tensor of shape (batch, seq) or (seq, batch)
    """
    range_vector = mnp.arange(0, max_length, 1, seq_length.dtype)
    result = range_vector < seq_length.view(seq_length.shape + (1,))
    if batch_first:
        return result.astype(mindspore.int64)
    return result.astype(mindspore.int64).swapaxes(0, 1)


class CRF(nn.Cell):
    """
    Linear-Chain Conditional Random Field layer.

    In training mode (tags provided):  returns the mean/sum NLL loss.
    In inference mode (no tags):       returns (score, history) for Viterbi decoding.

    Args:
        num_tags    : number of output tags
        batch_first : if True, input shape is (batch, seq, tags); else (seq, batch, tags)
        reduction   : loss reduction — 'none' | 'sum' | 'mean' | 'token_mean'
    """

    def __init__(self, num_tags: int, batch_first: bool = False, reduction: str = 'sum') -> None:
        if num_tags <= 0:
            raise ValueError(f'invalid number of tags: {num_tags}')
        super().__init__()
        if reduction not in ('none', 'sum', 'mean', 'token_mean'):
            raise ValueError(f'invalid reduction: {reduction}')

        self.num_tags    = num_tags
        self.batch_first = batch_first
        self.reduction   = reduction

        # Learnable start-of-sequence transition vector, shape (num_tags,)
        self.start_transitions = Parameter(
            initializer(Uniform(0.1), (num_tags,)), name='start_transitions'
        )
        # Learnable end-of-sequence transition vector, shape (num_tags,)
        self.end_transitions   = Parameter(
            initializer(Uniform(0.1), (num_tags,)), name='end_transitions'
        )
        # Learnable tag-to-tag transition matrix, shape (num_tags, num_tags)
        self.transitions       = Parameter(
            initializer(Uniform(0.1), (num_tags, num_tags)), name='transitions'
        )

    def construct(self, emissions, tags=None, seq_length=None):
        """
        Forward pass.

        If tags is None  → decode mode: return (score, history) via Viterbi.
        If tags provided → train mode:  return scalar NLL loss.
        """
        if tags is None:
            # Inference: run the Viterbi algorithm
            return self._decode(emissions, seq_length)
        # Training: compute negative log-likelihood loss
        return self._forward(emissions, tags, seq_length)

    def _forward(self, emissions, tags=None, seq_length=None):
        """Compute the CRF negative log-likelihood loss for a batch."""
        if self.batch_first:
            batch_size, max_length = tags.shape
            emissions = emissions.swapaxes(0, 1)
            tags      = tags.swapaxes(0, 1)
        else:
            max_length, batch_size = tags.shape

        if seq_length is None:
            seq_length = mnp.full((batch_size,), max_length, mindspore.int64)

        mask = sequence_mask(seq_length, max_length)

        # Numerator: score of the ground-truth path
        numerator   = compute_score(
            emissions, tags, seq_length - 1, mask,
            self.transitions, self.start_transitions, self.end_transitions
        )
        # Denominator: log-partition function (sum over all paths)
        denominator = compute_normalizer(
            emissions, mask,
            self.transitions, self.start_transitions, self.end_transitions
        )
        # NLL = log Z - Score(ground-truth)
        llh = denominator - numerator

        if self.reduction == 'none':        return llh
        if self.reduction == 'sum':         return llh.sum()
        if self.reduction == 'mean':        return llh.mean()
        # 'token_mean': normalise by total number of real tokens
        return llh.sum() / mask.astype(emissions.dtype).sum()

    def _decode(self, emissions, seq_length=None):
        """Run Viterbi decoding; returns (score, history) for post_decode."""
        if self.batch_first:
            batch_size, max_length = emissions.shape[:2]
            emissions = emissions.swapaxes(0, 1)
        else:
            batch_size, max_length = emissions.shape[:2]

        if seq_length is None:
            seq_length = mnp.full((batch_size,), max_length, mindspore.int64)

        mask = sequence_mask(seq_length, max_length)

        return viterbi_decode(
            emissions, mask,
            self.transitions, self.start_transitions, self.end_transitions
        )

## Section 2.5 — BiLSTM + CRF Model

The full model stacks four components:

1. **`nn.Embedding`** — maps token indices to dense vectors of size `embedding_dim`.
2. **`nn.LSTM`** (bidirectional) — extracts contextual features. Hidden size per direction is `hidden_dim // 2`, so the concatenated output is `hidden_dim`.
3. **`nn.Dense`** — projects LSTM outputs from `hidden_dim` to `num_tags` (emission scores).
4. **`CRF`** — computes NLL loss during training, or Viterbi scores during inference.

In [16]:
class BiLSTM_CRF(nn.Cell):
    """
    Bidirectional LSTM + CRF model for Named Entity Recognition.

    Args:
        vocab_size    : size of the token vocabulary
        embedding_dim : dimensionality of token embeddings
        hidden_dim    : total hidden size of the BiLSTM
                        (each direction uses hidden_dim // 2 units)
        num_tags      : number of NER output tags
        padding_idx   : vocabulary index used for padding (default 0)
    """

    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_tags, padding_idx=0):
        super().__init__()

        # 1. Embedding layer: vocab_size × embedding_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)

        # 2. Bidirectional LSTM: each direction has hidden_dim//2 units
        #    Output shape: (batch, seq, hidden_dim)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim // 2,
            bidirectional=True,
            batch_first=True
        )

        # 3. Linear projection from hidden_dim → num_tags (emission scores)
        self.hidden2tag = nn.Dense(hidden_dim, num_tags, 'he_uniform')

        # 4. CRF layer (batch_first=True to match LSTM output format)
        self.crf = CRF(num_tags, batch_first=True)

    def construct(self, inputs, seq_length, tags=None):
        """
        Forward pass.

        Args:
            inputs     : Tensor (batch, seq) int64  — token index sequences
            seq_length : Tensor (batch,)     int64  — true sequence lengths
            tags       : Tensor (batch, seq) int64  — ground-truth labels (train only)

        Returns:
            Training  : scalar loss (NLL)
            Inference : (score, history) tuple for post_decode
        """
        # Step 1: look up token embeddings
        # shape: (batch, seq, embedding_dim)
        embeds = self.embedding(inputs)

        # Step 2: pass through bidirectional LSTM
        # outputs shape: (batch, seq, hidden_dim)
        outputs, _ = self.lstm(embeds, seq_length=seq_length)

        # Step 3: project to tag emission scores
        # shape: (batch, seq, num_tags)
        feats = self.hidden2tag(outputs)

        # Step 4: pass emission scores (and optional tags) through CRF
        crf_outs = self.crf(feats, tags, seq_length)

        return crf_outs

---
# Task 3: Model Training and Saving

## Section 3.1 — Instantiate the Model

We choose small hyperparameters (`embedding_dim=16`, `hidden_dim=32`) so the model trains quickly on CPU as well as GPU.

In [17]:
# Hyperparameters
embedding_dim = 16   # dimension of token embeddings
hidden_dim    = 32   # total BiLSTM hidden size (16 per direction)

In [18]:
# Instantiate the BiLSTM-CRF model
model = BiLSTM_CRF(
    vocab_size    = len(word_to_idx),
    embedding_dim = embedding_dim,
    hidden_dim    = hidden_dim,
    num_tags      = len(tag_to_idx)
)

## Section 3.2 — Optimizer

We use the **Adam** optimiser with a learning rate of 0.001 and no weight decay. Adam generally converges faster than plain SGD for NLP tasks.

An alternative SGD-with-momentum setup is shown commented out below.

In [19]:
# Adam optimizer — recommended for sequence labelling tasks
optimizer = nn.Adam(model.trainable_params(), learning_rate=0.001, weight_decay=0.0)

# Alternatively, uncomment to use SGD with weight decay:
# optimizer = nn.SGD(model.trainable_params(), learning_rate=0.001, weight_decay=1e-4)

## Section 3.3 — Training Loop with Mini-Batches

We use MindSpore's **functional differentiation** API (`ops.value_and_grad`) rather than the high-level `Model.train` wrapper. This gives us fine-grained control over the training loop.

The training procedure:
1. Shuffle all 14 987 samples into mini-batches of size 8.
2. For each batch, compute the forward pass (CRF NLL loss) and back-propagate gradients.
3. Repeat for 5 epochs.

> **Note**: The CRF layer already outputs the NLL loss, so no separate loss function is needed.

In [20]:
import mindspore.ops as ops

# Create a function that computes both the loss value AND its gradients
# with respect to the model's trainable parameters in a single call.
grad_fn = mindspore.ops.value_and_grad(model, None, optimizer.parameters)


def train_step(data, seq_length, label):
    """
    Execute one gradient-update step on a mini-batch.

    Args:
        data       : Tensor (batch, seq)  — token indices
        seq_length : Tensor (batch,)      — true lengths
        label      : Tensor (batch, seq)  — ground-truth NER tags

    Returns:
        loss : scalar Tensor — CRF NLL loss for this batch
    """
    # Forward pass + compute gradients in one shot
    loss, grads = grad_fn(data, seq_length, label)
    # Apply the gradients to update the model parameters
    optimizer(grads)
    return loss

In [21]:
# Pre-convert the entire training set to tensors once (avoids repeated conversion)
data, label, seq_length = prepare_sequence(training_data, word_to_idx, tag_to_idx)
len(data)

14987

In [22]:
import sys
print(sys.version)

3.7.10 | packaged by conda-forge | (default, Oct 13 2021, 21:01:18) 
[GCC 9.4.0]


In [ ]:
from tqdm import tqdm

# ── Training hyperparameters ─────────────────────────────────────────────────
batch_size = 8    # samples per gradient update
epochs     = 5    # full passes over the training data

# Total number of gradient steps across all epochs
steps = (len(training_data) // batch_size) * epochs

# ── Training loop ────────────────────────────────────────────────────────────
with tqdm(total=steps) as t:
    for epoch in range(epochs):
        for i in range(len(training_data) // batch_size):
            # Slice the i-th mini-batch
            batch_data   = data[       i * batch_size : (i + 1) * batch_size]
            batch_len    = seq_length[ i * batch_size : (i + 1) * batch_size]
            batch_labels = label[      i * batch_size : (i + 1) * batch_size]

            loss = train_step(batch_data, batch_len, batch_labels)

            # Update progress bar with current loss
            t.set_postfix(epoch=epoch + 1, loss=float(loss.asnumpy()))
            t.update(1)

100%|██████████| 9365/9365 [2:09:36<00:00,  1.20it/s, epoch=5, loss=nan]  


## Section 3.4 — Save the Trained Model

MindSpore serialises model parameters into a `.ckpt` (checkpoint) file. This file stores all `Parameter` tensors and can be reloaded with `load_checkpoint` + `load_param_into_net`.

In [26]:
import mindspore as ms

# Save all trainable parameters to a checkpoint file
ms.save_checkpoint(model, "./LSTM_CRFNet.ckpt")
print("Model saved to LSTM_CRFNet.ckpt")

Model saved to LSTM_CRFNet.ckpt


## Section 3.5 — Export the Model to AIR Format

MindSpore's `export` function serialises the model's **static computation graph** into a portable file. The **AIR** (Ascend IR) format is the standard exchange format for deployment on Ascend hardware and with MindSpore Serving.

We reload the checkpoint into a fresh model instance before exporting, which is the recommended practice to ensure the exported graph matches the saved weights exactly.

In [27]:
from mindspore import export, load_checkpoint, load_param_into_net
from mindspore import Tensor
import numpy as np

# Instantiate a fresh model with the same architecture
model = BiLSTM_CRF(len(word_to_idx), embedding_dim, hidden_dim, len(tag_to_idx))

# Step 1: Load checkpoint parameters into a dictionary
param_dict = load_checkpoint("LSTM_CRFNet.ckpt")

# Step 2: Transfer the loaded parameters into the fresh model
load_param_into_net(model, param_dict)

# Step 3: Export the model to MINDIR format
# `data` and `seq_length` serve as example inputs that define the input shapes
# for the static graph compilation
export(model, data, seq_length, file_name='LSTM_CRF', file_format='MINDIR')
print("Model exported to LSTM_CRF.mindir")

Model exported to LSTM_CRF.mindir


---
# Task 4: Model Prediction

We now load the trained model and run inference on the test split of CoNLL-2003. The prediction pipeline is:

1. Load test data from `test.txt`
2. Build a vocabulary from the test sentences and apply the same label encoding
3. Pass token tensors through the model in decode mode (no `tags` argument)
4. Apply `post_decode` to convert Viterbi scores to tag index sequences
5. Map indices back to human-readable tag strings

## Section 4.1 — Load the Saved Model

In [28]:
from mindspore import load_checkpoint, load_param_into_net

# Create a fresh model instance
model = BiLSTM_CRF(len(word_to_idx), embedding_dim, hidden_dim, len(tag_to_idx))

# Load saved parameters from the checkpoint file
param_dict = load_checkpoint("LSTM_CRFNet.ckpt")

# Inject the saved parameters into the model (returns a list of missing/unexpected keys)
not_loaded = load_param_into_net(model, param_dict)
not_loaded   # should be [] if all parameters loaded successfully

([], [])

## Section 4.2 — Prepare the Test Data

In [29]:
# Load the test split (same format as train.txt)
test_sentences, test_labels = load_file("/test.txt")

In [30]:
def get_dataset(sentences, labels):
    """
    Zip test sentences and labels into (sentence, label) tuples.
    The same helper used for training data.
    """
    dataset = []
    for i in range(len(sentences)):
        data_sample = tuple((test_sentences[i], test_labels[i]))
        dataset.append(data_sample)
    return dataset

# Use only the first 4 test sentences for a quick demonstration
test_data = get_dataset(test_sentences[:4], test_labels)
len(test_data)

4

In [31]:
# Build a vocabulary from the test sentences
# (In production you would reuse the training vocabulary to avoid OOV issues;
#  here we rebuild it from test data to keep the demo self-contained.)
word_to_idx = {}
word_to_idx['<pad>'] = 0

for sentence, tags in test_data:
    for word in sentence:
        if word not in word_to_idx:
            word_to_idx[word] = len(word_to_idx)

# Reuse the same label encoding as during training
tag_to_idx = {
    "B-PER": 0, "I-PER": 1,
    "B-ORG": 2, "I-ORG": 3,
    "B-LOC": 4, "I-LOC": 5,
    "B-MISC": 6, "I-MISC": 7,
    "O": 8
}

In [32]:
# Convert the test sentences to padded/truncated tensors
data, label, seq_length = prepare_sequence(test_data, word_to_idx, tag_to_idx)

## Section 4.3 — Run Model Inference

In [33]:
# Call the model without labels → decode mode (Viterbi)
# Returns:
#   score   : Tensor (batch_size, num_tags) — best-path scores at the last step
#   history : tuple of back-pointer Tensors  — one per time step
score, history = model(data, seq_length)
score

Tensor(shape=[4, 9], dtype=Float32, value=
[[            nan,             nan,             nan ...             nan,             nan,             nan],
 [            nan,             nan,             nan ...             nan,             nan,             nan],
 [            nan,             nan,             nan ...             nan,             nan,             nan],
 [            nan,             nan,             nan ...             nan,             nan,             nan]])

In [34]:
# Apply the Viterbi back-tracking post-processor to recover the best tag sequences
# predict is a list of lists of integer tag indices, one list per test sample
predict = post_decode(score, history, seq_length)
predict

[[0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0], [0, 0, 0, 0, 0, 0]]

In [35]:
# Build reverse mapping: integer index → tag string
idx_to_tag = {idx: tag for tag, idx in tag_to_idx.items()}


def sequence_to_tag(sequences, idx_to_tag):
    """
    Convert a list of integer index sequences to human-readable NER tag sequences.

    Args:
        sequences  : list[list[int]] — predicted tag indices
        idx_to_tag : dict[int, str]  — reverse tag mapping

    Returns:
        list[list[str]] — predicted tag strings
    """
    outputs = []
    for seq in sequences:
        outputs.append([idx_to_tag[i] for i in seq])
    return outputs

In [36]:
# Display the final predicted NER tags for each test sentence
predicted_tags = sequence_to_tag(predict, idx_to_tag)

for i, (sentence, true_tags, pred_tags) in enumerate(
    zip(test_sentences[:4], test_labels[:4], predicted_tags)
):
    print(f"\n── Sample {i+1} ───────────────────────")
    print(f"  Tokens    : {sentence[:len(pred_tags)]}")
    print(f"  True tags : {true_tags[:len(pred_tags)]}")
    print(f"  Pred tags : {pred_tags}")


── Sample 1 ───────────────────────
  Tokens    : ['-DOCSTART-']
  True tags : ['O']
  Pred tags : ['B-PER']

── Sample 2 ───────────────────────
  Tokens    : ['SOCCER', '-', 'JAPAN', 'GET', 'LUCKY', 'WIN', ',', 'CHINA', 'IN', 'SURPRISE']
  True tags : ['O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'B-PER', 'O', 'O']
  Pred tags : ['B-PER', 'B-PER', 'B-PER', 'B-PER', 'B-PER', 'B-PER', 'B-PER', 'B-PER', 'B-PER', 'B-PER']

── Sample 3 ───────────────────────
  Tokens    : ['Nadim', 'Ladki']
  True tags : ['B-PER', 'I-PER']
  Pred tags : ['B-PER', 'B-PER']

── Sample 4 ───────────────────────
  Tokens    : ['AL-AIN', ',', 'United', 'Arab', 'Emirates', '1996-12-06']
  True tags : ['B-LOC', 'O', 'B-LOC', 'I-LOC', 'I-LOC', 'O']
  Pred tags : ['B-PER', 'B-PER', 'B-PER', 'B-PER', 'B-PER', 'B-PER']
